In [1]:
import pandas as pd 
import numpy as np 
import parquet
import seaborn as sns 
import matplotlib.pyplot as plt 

In [55]:
df = pd.read_parquet('bikeshare_all.parquet')
# df.head()

In [56]:
before_mem = df.memory_usage(deep=True).sum() / 1e9
before_mem

np.float64(2.232043515)

In [57]:
df.dtypes

trip_id                        int64
trip_duration                float32
start_station_id             float32
start_time            datetime64[us]
start_station_name          category
end_station_id               float32
end_time              datetime64[us]
end_station_name            category
bike_id                      float32
user_type                   category
bike_model                  category
trip__duration               float32
dtype: object

In [ ]:
# bug from the previous step
df[~df['trip__duration'].isna()]['end_time'].dt.year.unique()

array([2019, 2020], dtype=int32)

In [62]:
df['trip_duration'] = df['trip_duration'].combine_first(df['trip__duration'])

insert NA trip_duration if end_time and start_time are known 

In [6]:
# sum(df['trip_duration'].isna() & ~df['end_time'].isna() & ~df['start_time'].isna())

2439766

In [63]:
df['trip_duration_calc'] = (df['end_time'] - df['start_time']).dt.total_seconds()

In [64]:
df.loc[df.trip_duration!=df.trip_duration_calc, ['trip_duration', 'trip_duration_calc', 'end_time', 'start_time']].head()

,trip_duration,trip_duration_calc,end_time,start_time
3821,0.0,NaN,NaT,2026-01-02 14:38:08
6965,0.0,NaN,NaT,2026-01-03 11:30:50
7364,0.0,NaN,NaT,2026-01-03 13:05:01
18005,0.0,NaN,NaT,2026-01-06 08:17:55
19260,0.0,NaN,NaT,2026-01-06 10:05:57


In [36]:
df.loc[df.trip_duration!=df.trip_duration_calc, ['trip_duration', 'trip_duration_calc', 'end_time', 'start_time']].tail()

,trip_duration,trip_duration_calc,end_time,start_time
44638686,332.0,300.0,2016-09-30 23:41:00,2016-09-30 23:36:00
44638687,4234.0,4260.0,2016-10-01 00:51:00,2016-09-30 23:40:00
44638688,3834.0,3840.0,2016-10-01 00:51:00,2016-09-30 23:47:00
44638689,470.0,480.0,2016-10-01 00:00:00,2016-09-30 23:52:00
44638690,1400.0,1380.0,2016-10-01 00:19:00,2016-09-30 23:56:00


In [65]:
# let's skip all trip_duration==0, probably some technical issues, outside my analysis
df = df.loc[df['trip_duration']!=0]

In [66]:
df.drop(columns=['trip__duration'], inplace=True)

In [68]:
df.memory_usage(deep=True).sum()/1e9

np.float64(2.764944335)

In [70]:
# df.dtypes

In [40]:
df.loc[df.trip_duration!=df.trip_duration_calc, ['trip_duration', 'trip_duration_calc', 'end_time', 'start_time']]

,trip_duration,trip_duration_calc,end_time,start_time
285185,1196.0,1197.0,2026-03-05 14:11:10,2026-03-05 13:51:13
291959,907.0,908.0,2026-03-06 13:47:41,2026-03-06 13:32:33
294793,916.0,917.0,2026-03-06 18:42:50,2026-03-06 18:27:33
304552,4788.0,8388.0,2026-03-08 03:05:04,2026-03-08 00:45:16
304555,4706.0,8306.0,2026-03-08 03:04:37,2026-03-08 00:46:11
...,...,...,...,...
44638686,332.0,300.0,2016-09-30 23:41:00,2016-09-30 23:36:00
44638687,4234.0,4260.0,2016-10-01 00:51:00,2016-09-30 23:40:00
44638688,3834.0,3840.0,2016-10-01 00:51:00,2016-09-30 23:47:00
44638689,470.0,480.0,2016-10-01 00:00:00,2016-09-30 23:52:00


In [50]:
df.loc[df['trip_duration'].isna() & ~df['trip_duration_calc'].isna()].shape

(2439766, 13)

In [51]:
df.loc[~df['trip_duration'].isna() & df['trip_duration_calc'].isna()].shape

(0, 13)

In [52]:
# it's better to get rid of trip_duration and work with trip_duration_calc
df.drop(columns='trip_duration', inplace=True)

In [ ]:
# let's check maybe there is some negatives:
df.loc[df['trip_duration_calc']<0].shape

(416, 12)

In [ ]:
df.loc[df['trip_duration_calc']<0]['end_time'].dt.year.unique()
# 2025, 2024, 2023, 2022, 2021, 2020, 2019, 2018, 2017 we have this issue with end_date<start_date for every year except 2026

array([2025, 2024, 2023, 2022, 2021, 2020, 2019, 2018, 2017], dtype=int32)

In [ ]:
df.loc[df['trip_duration_calc']<0]['end_time'].dt.month.unique() #all in november. 

array([11], dtype=int32)

In [ ]:
df.loc[df['trip_duration_calc']<0]['end_time'].dt.day.unique() # all in the first week

array([2, 3, 5, 6, 7, 1, 4], dtype=int32)

In [90]:
df.loc[df['trip_duration_calc']<0]['end_time'].dt.to_period('M').dt.to_timestamp()

8900087    2025-11-01
8900119    2025-11-01
8900198    2025-11-01
8900210    2025-11-01
8900415    2025-11-01
              ...    
43979060   2017-11-01
43979063   2017-11-01
43979065   2017-11-01
43979066   2017-11-01
43979067   2017-11-01
Name: end_time, Length: 416, dtype: datetime64[us]

In [ ]:
# Formula for calendar-aligned week of the month
first_weekdays = df.loc[df['trip_duration_calc']<0]['end_time'].dt.to_period('M').dt.to_timestamp().dt.weekday
sum(((df.loc[df['trip_duration_calc']<0]['end_time'].dt.day + first_weekdays - 1) // 7) + 1==1)
# so it's probably clocks roll back an hour

416

In [109]:

df.loc[(df['trip_duration'].isna()) & (df['trip_duration_calc']<0)] 

,trip_id,trip_duration,start_station_id,start_time,start_station_name,end_station_id,end_time,end_station_name,bike_id,user_type,bike_model,trip_duration_calc


In [ ]:
# fill trip_duration_calc with trip_duration and rebuild end_time with info from trip_duration:
 
mask = df['trip_duration_calc'] < 0
df.loc[mask, 'end_time'] = df.loc[mask, 'start_time'] + pd.to_timedelta(df.loc[mask, 'trip_duration'], unit='s')
df.loc[mask, 'trip_duration_calc'] = df.loc[mask, 'trip_duration']

In [114]:
df.loc[~df['trip_duration'].isna() & df['trip_duration_calc'].isna()].shape[0]

0

In [122]:
(df.loc[df['trip_duration'] != df['trip_duration_calc'], ['trip_duration', 'trip_duration_calc']] \
    .assign(diff = lambda d: (d['trip_duration_calc'] - d['trip_duration'])/60)
    .sort_values('diff', ascending=False)
    .head()
    )

,trip_duration,trip_duration_calc,diff
40254389,12294.0,612300.0,10000.100000
38822881,760.0,6780.0,100.333333
38822894,716.0,6720.0,100.066667
39333776,5.0,4020.0,66.916667
39367709,450.0,4440.0,66.500000


In [ ]:
# i'll stay with trip_duration, fill it with trip_duration_calc where trip_duration.isna() and trip_duration_calc known 
# remove trips with trip_duration_calc, probably technical issue.

In [124]:
df['trip_duration'] = df['trip_duration'].combine_first(df['trip_duration_calc'])

In [125]:
df = df.loc[df['trip_duration']!=0]

In [127]:
df.drop(columns='trip_duration_calc', inplace=True)

In [130]:
df.memory_usage(deep=True).sum()/1e9

np.float64(2.586567039)

### duplicates check

In [128]:
dup_ids = df.loc[df.duplicated(subset=['trip_id']), 'trip_id']
pairs = df[df['trip_id'].isin(dup_ids)].sort_values('trip_id')
print(pairs.shape)
pairs[['trip_id', 'trip_duration', 'start_time', 'end_time', 'start_station_name', 'end_station_name', 'bike_id']]


(0, 11)


,trip_id,trip_duration,start_time,end_time,start_station_name,end_station_name,bike_id


In [131]:
df.head()

,trip_id,trip_duration,start_station_id,start_time,start_station_name,end_station_id,end_time,end_station_name,bike_id,user_type,bike_model
0,43682415,248.0,7719.0,2026-01-01 00:00:26,Wolseley St / Augusta Ave,7037.0,2026-01-01 00:04:34,Bathurst St / Dundas St W,122.0,Member,ICONIC
1,43682417,295.0,8001.0,2026-01-01 00:02:05,Union Station (North) - SMART,7076.0,2026-01-01 00:07:00,York St / Queens Quay W,4900.0,Member,ICONIC
2,43682418,2701.0,7172.0,2026-01-01 00:04:15,Strachan Ave / Princes' Blvd,7100.0,2026-01-01 00:49:16,Dundas St E / Regent Park Blvd,1833.0,Member,ICONIC
3,43682419,581.0,7259.0,2026-01-01 00:04:29,Lower Spadina Ave / Lake Shore Blvd W,7076.0,2026-01-01 00:14:10,York St / Queens Quay W,5078.0,Casual,ICONIC
4,43682420,600.0,7259.0,2026-01-01 00:04:53,Lower Spadina Ave / Lake Shore Blvd W,7076.0,2026-01-01 00:14:53,York St / Queens Quay W,3951.0,Member,ICONIC


In [132]:
df.to_parquet('bikeshare_clean.parquet', index=False)